# TelcoRetrieve — Phase 2: real-generator GPU run (60-question validation set)

Fixed generator **google/gemma-3n-E4B-it** served by vLLM on :8000, fixed
10 GB index (`otel-0.6b_text_baseline_b512_v256_full`), official CLI
`open-telco-rag challenge export` + `check` + `score`.

Variants (all on the official pipeline: hybrid retrieval, top-k 10,
OTel reranker, temperature 0, max 8 answer tokens, seed 1234):
- **control** — official defaults (rerank keeps 5)
- **b850** — defaults + budgeted context fill to 850 passage tokens
  (`--prompt-builder my_hooks:budget850`)
- **k3** — defaults + `--rerank-top-k 3`

NOTE on the phase-1 "256/k3" variant: the fixed index chunks are
**≤512 tokens** (text_baseline, bound_tokens 512) — there is no 256-token
collection in the official index, so the 256-token chunk size cannot be
reproduced here. `k3` is the closest faithful analogue on the fixed index:
fewer passages at the fixed 512-token chunk size.

Budget: ≤6 GPU-hours for this first pass. The session wall time is the
GPU time (T4 x2); it is printed at the end.


In [ ]:
import os, time
T0 = time.time()
WORK = "/kaggle/working"
P2 = f"{WORK}/phase2"
os.makedirs(P2, exist_ok=True)
open(f"{P2}/t0.txt", "w").write(str(T0))
print("WORK:", WORK)
!nvidia-smi --query-gpu=name,memory.total --format=csv
!df -h /kaggle/working | tail -1
!python3 -c "import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| gpus', torch.cuda.device_count())"


In [ ]:
# Install the official CLI (with local embedding) + vLLM. ~5-15 min.
!pip install -q "open-telco-rag[local-embed] @ git+https://github.com/Znbne/telco-retrieve" vllm 2>&1 | tail -3
!open-telco-rag challenge export --help | head -8
!python3 -c "import vllm, torch; print('vllm', vllm.__version__, '| torch', torch.__version__)"
!python3 -c "import sentence_transformers, chromadb, tiktoken; print('sentence-transformers', sentence_transformers.__version__)"


In [ ]:
# Write the input files (embedded as base64 so the notebook is self-contained).
import base64, pathlib
FILES = {
    "validation.csv": "...",
    "validation_answers.csv": "IklEIiwiYW5zd2VyIgoidmFsLTAwMSIsIkEiCiJ2YWwtMDAyIiwiQSIKInZhbC0wMDMiLCJBIgoidmFsLTAwNCIsIkEiCiJ2YWwtMDA1IiwiRCIKInZhbC0wMDYiLCJBIgoidmFsLTAwNyIsIlRSVUUiCiJ2YWwtMDA4IiwiVFJVRSIKInZhbC0wMDkiLCJGQUxTRSIKInZhbC0wMTAiLCJGQUxTRSIKInZhbC0wMTEiLCJUUlVFIgoidmFsLTAxMiIsIkZBTFNFIgoidmFsLTAxMyIsIkEiCiJ2YWwtMDE0IiwiQiIKInZhbC0wMTUiLCJDIgoidmFsLTAxNiIsIkQiCiJ2YWwtMDE3IiwiQSIKInZhbC0wMTgiLCJCIgoidmFsLTAxOSIsIkMiCiJ2YWwtMDIwIiwiRCIKInZhbC0wMjEiLCJBIgoidmFsLTAyMiIsIkIiCiJ2YWwtMDIzIiwiQyIKInZhbC0wMjQiLCJEIgoidmFsLTAyNSIsIlRSVUUiCiJ2YWwtMDI2IiwiRkFMU0UiCiJ2YWwtMDI3IiwiVFJVRSIKInZhbC0wMjgiLCJGQUxTRSIKInZhbC0wMjkiLCJUUlVFIgoidmFsLTAzMCIsIkZBTFNFIgoidmFsLTAzMSIsIlRSVUUiCiJ2YWwtMDMyIiwiRkFMU0UiCiJ2YWwtMDMzIiwiVFJVRSIKInZhbC0wMzQiLCJGQUxTRSIKInZhbC0wMzUiLCJUUlVFIgoidmFsLTAzNiIsIkZBTFNFIgoidmFsLTAzNyIsIkEiCiJ2YWwtMDM4IiwiQiIKInZhbC0wMzkiLCJDIgoidmFsLTA0MCIsIkQiCiJ2YWwtMDQxIiwiQSIKInZhbC0wNDIiLCJCIgoidmFsLTA0MyIsIkMiCiJ2YWwtMDQ0IiwiRCIKInZhbC0wNDUiLCJBIgoidmFsLTA0NiIsIkIiCiJ2YWwtMDQ3IiwiQyIKInZhbC0wNDgiLCJEIgoidmFsLTA0OSIsIlRSVUUiCiJ2YWwtMDUwIiwiRkFMU0UiCiJ2YWwtMDUxIiwiVFJVRSIKInZhbC0wNTIiLCJGQUxTRSIKInZhbC0wNTMiLCJUUlVFIgoidmFsLTA1NCIsIkZBTFNFIgoidmFsLTA1NSIsIlRSVUUiCiJ2YWwtMDU2IiwiRkFMU0UiCiJ2YWwtMDU3IiwiVFJVRSIKInZhbC0wNTgiLCJGQUxTRSIKInZhbC0wNTkiLCJUUlVFIgoidmFsLTA2MCIsIkZBTFNFIgo=",
    "my_hooks.py": "IiIiUHJvbXB0LWJ1aWxkZXIgaG9va3MgZm9yIHRoZSBUZWxjb1JldHJpZXZlIHBoYXNlLTIgS2FnZ2xlIEdQVSBydW4uCgpidWRnZXQ4NTA6IGJ1ZGdldGVkIGNvbnRleHQgZmlsbCB0byA4NTAgcGFzc2FnZSB0b2tlbnMuIE1pcnJvcnMgdGhlIGxvY2FsCnNjYWZmb2xkJ3MgYnVpbGRfcHJvbXB0X2J1ZGdldGVkIChhbmQgdXBzdHJlYW0gYGV2YWx1YXRlYAotLWNvbnRleHQtdG9rZW4tYnVkZ2V0KToga2VlcCBwYXNzYWdlcyBpbiByYW5rIG9yZGVyIHdoaWxlIHRoZWlyIGN1bXVsYXRpdmUKY2wxMDBrX2Jhc2UgdG9rZW4gY291bnQgc3RheXMgd2l0aGluIHRoZSBidWRnZXQ7IGFsd2F5cyBrZWVwIGF0IGxlYXN0IHRoZQp0b3AtcmFua2VkIHBhc3NhZ2U7IHRoZW4gYnVpbGQgdGhlIHByb21wdCAod2hpY2ggYWxzbyBlbmZvcmNlcyB0aGUgb2ZmaWNpYWwKNDA5Ni10b2tlbiBjYXApLiBUaGUgYnVkZ2V0IGNvdW50cyBwYXNzYWdlIHRleHQgb25seSDigJQgc3lzdGVtIG1lc3NhZ2UsCnF1ZXN0aW9uIGFuZCBvcHRpb25zIGFyZSBmaXhlZCBvdmVyaGVhZCBjb3VudGVkIGluIHRoZSBwcm9tcHQgdG90YWwuCgpVc2FnZSAocnVuIGV4cG9ydCBmcm9tIHRoZSBkaXJlY3RvcnkgaG9sZGluZyB0aGlzIGZpbGUpOgogICAgb3Blbi10ZWxjby1yYWcgY2hhbGxlbmdlIGV4cG9ydCAuLi4gLS1wcm9tcHQtYnVpbGRlciBteV9ob29rczpidWRnZXQ4NTAKIiIiCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmZyb20gb3Blbl90ZWxjb19yYWcuY2hhbGxlbmdlLnByb21wdCBpbXBvcnQgYnVpbGRfcHJvbXB0CmZyb20gb3Blbl90ZWxjb19yYWcuY2hhbGxlbmdlLnRva2VucyBpbXBvcnQgY291bnRfdG9rZW5zCgpCVURHRVRfVE9LRU5TID0gODUwCgoKZGVmIGJ1ZGdldDg1MChxdWVzdGlvbiwgcGFzc2FnZXMsIG1heF9wcm9tcHRfdG9rZW5zKToKICAgIGtlcHQgPSBbXQogICAgdXNlZCA9IDAKICAgIGZvciBwYXNzYWdlIGluIHBhc3NhZ2VzOgogICAgICAgIG4gPSBjb3VudF90b2tlbnMocGFzc2FnZS50ZXh0KQogICAgICAgIGlmIGtlcHQgYW5kIHVzZWQgKyBuID4gQlVER0VUX1RPS0VOUzoKICAgICAgICAgICAgYnJlYWsKICAgICAgICBrZXB0LmFwcGVuZChwYXNzYWdlKQogICAgICAgIHVzZWQgKz0gbgogICAgcmV0dXJuIGJ1aWxkX3Byb21wdChxdWVzdGlvbiwga2VwdCwgbWF4X3Byb21wdF90b2tlbnMpCg==",
}
for name, blob in FILES.items():
    p = pathlib.Path("/kaggle/working/phase2") / name
    p.write_bytes(base64.b64decode(blob))
    print(name, p.stat().st_size, "bytes")
!ls -la /kaggle/working/phase2/


In [ ]:
# Pull the fixed 10 GB index once (cached for all 3 variants in this session).
!cd /kaggle/working && open-telco-rag pull-hf GSMA/telco-retrieve-chunks telcoretrievechallenge/fixed-grid/text_baseline/otel-0.6b --local-dir /kaggle/working/runs 2>&1 | tail -5
!ls /kaggle/working/runs/telcoretrievechallenge/fixed-grid/text_baseline/otel-0.6b/db/
!df -h /kaggle/working | tail -1


In [ ]:
# Serve Gemma 3n E4B-it with vLLM (model attached from the Kaggle model hub).
import glob, os, subprocess, time, urllib.request, json
cands = glob.glob("/kaggle/input/**/config.json", recursive=True)
print("config.json candidates:", cands)
MODEL_DIR = os.path.dirname([c for c in cands if "gemma-3n-e4b-it" in c][0])
print("MODEL_DIR:", MODEL_DIR)
LOG = "/kaggle/working/phase2/vllm.log"
cmd = ["python", "-m", "vllm.entrypoints.openai.api_server",
       "--model", MODEL_DIR,
       "--served-model-name", "google/gemma-3n-E4B-it",
       "--port", "8000", "--max-model-len", "8192",
       "--tensor-parallel-size", "2", "--dtype", "bfloat16",
       "--gpu-memory-utilization", "0.70", "--enforce-eager",
       "--max-num-seqs", "4", "--disable-log-stats"]
logf = open(LOG, "w")
proc = subprocess.Popen(cmd, stdout=logf, stderr=subprocess.STDOUT)
print("vLLM pid:", proc.pid)
deadline = time.time() + 20 * 60
up = False
while time.time() < deadline:
    try:
        with urllib.request.urlopen("http://localhost:8000/v1/models", timeout=5) as r:
            data = json.loads(r.read())
            print("server up:", [m["id"] for m in data.get("data", [])])
            up = True
            break
    except Exception:
        time.sleep(15)
if not up:
    print("VLLM FAILED TO START — last 100 log lines:")
    print("\n".join(open(LOG).read().splitlines()[-100:]))
    raise SystemExit("vLLM did not come up in 20 min")


In [ ]:
# Smoke-test generation (expect a TRUE/FALSE letter back).
import json, urllib.request
payload = {"model": "google/gemma-3n-E4B-it",
           "messages": [{"role": "system", "content": "You answer 3GPP questions from the provided context. Reply with only the answer."},
                        {"role": "user", "content": "Passages:\nNone\n\nQuestion: Is the sky blue?\n\nAnswer with only TRUE or FALSE."}],
           "temperature": 0, "max_tokens": 8}
req = urllib.request.Request("http://localhost:8000/v1/chat/completions",
    data=json.dumps(payload).encode(), headers={"Content-Type": "application/json"})
with urllib.request.urlopen(req, timeout=120) as r:
    out = json.loads(r.read())
print("smoke reply:", repr(out["choices"][0]["message"]["content"]))


In [ ]:
# Variant 1/3 — control: official defaults (hybrid, top-k 10, rerank keeps 5).
!cd /kaggle/working/phase2 && time LOCAL_LLM_BASE_URL=http://localhost:8000/v1 open-telco-rag challenge export --questions /kaggle/working/phase2/validation.csv --out /kaggle/working/phase2/control/submission.csv --db-path /kaggle/working/runs/telcoretrievechallenge/fixed-grid/text_baseline/otel-0.6b/db --collection otel-0.6b_text_baseline_b512_v256_full --timeout 300 --resume


In [ ]:
# Variant 2/3 — b850: defaults + budgeted context fill to 850 passage tokens.
!cd /kaggle/working/phase2 && time LOCAL_LLM_BASE_URL=http://localhost:8000/v1 open-telco-rag challenge export --questions /kaggle/working/phase2/validation.csv --out /kaggle/working/phase2/b850/submission.csv --db-path /kaggle/working/runs/telcoretrievechallenge/fixed-grid/text_baseline/otel-0.6b/db --collection otel-0.6b_text_baseline_b512_v256_full --timeout 300 --resume --prompt-builder my_hooks:budget850


In [ ]:
# Variant 3/3 — k3: defaults + rerank keeps 3 (fewer passages, fixed 512-token chunks).
!cd /kaggle/working/phase2 && time LOCAL_LLM_BASE_URL=http://localhost:8000/v1 open-telco-rag challenge export --questions /kaggle/working/phase2/validation.csv --out /kaggle/working/phase2/k3/submission.csv --db-path /kaggle/working/runs/telcoretrievechallenge/fixed-grid/text_baseline/otel-0.6b/db --collection otel-0.6b_text_baseline_b512_v256_full --timeout 300 --resume --rerank-top-k 3


In [ ]:
# Check + score each variant with the official metrics.
!cd /kaggle/working/phase2 && for v in control b850 k3; do echo "===== $v : check ====="; open-telco-rag challenge check --submission /kaggle/working/phase2/$v/submission.csv --questions /kaggle/working/phase2/validation.csv; echo "===== $v : score (official) ====="; open-telco-rag challenge score --submission /kaggle/working/phase2/$v/submission.csv --reference /kaggle/working/phase2/validation_answers.csv; done


In [ ]:
# Independent summary: accuracy (overall / MC / TF), token efficiency,
# overall score, tok/q — recomputed from the submission CSVs (cross-checks
# the official `challenge score` numbers above).
import csv, json
import tiktoken
enc = tiktoken.get_encoding("cl100k_base")
ref = {r["ID"]: r["answer"] for r in csv.DictReader(open("/kaggle/working/phase2/validation_answers.csv"))}
qtype = {}
for r in csv.DictReader(open("/kaggle/working/phase2/validation.csv")):
    qtype[r["ID"]] = "TF" if not r["options"] else "MC"
summary = {}
for variant in ("control", "b850", "k3"):
    rows = list(csv.DictReader(open(f"/kaggle/working/phase2/{variant}/submission.csv")))
    total_tokens = sum(len(enc.encode(r["input_prompt"], disallowed_special=())) for r in rows)
    n = len(rows)
    correct = sum(1 for r in rows if r["answer"] == ref.get(r["ID"]))
    mc = [r for r in rows if qtype[r["ID"]] == "MC"]
    tf = [r for r in rows if qtype[r["ID"]] == "TF"]
    acc_mc = 100 * sum(1 for r in mc if r["answer"] == ref.get(r["ID"])) / max(1, len(mc))
    acc_tf = 100 * sum(1 for r in tf if r["answer"] == ref.get(r["ID"])) / max(1, len(tf))
    acc = 100 * correct / n
    eff = 100 * 850 * n / (850 * n + total_tokens)
    overall = 0.7 * acc + 0.3 * eff
    summary[variant] = {
        "n": n, "accuracy_pct": round(acc, 2), "accuracy_mc_pct": round(acc_mc, 2),
        "accuracy_tf_pct": round(acc_tf, 2), "token_efficiency_pct": round(eff, 2),
        "overall": round(overall, 2), "total_prompt_tokens": total_tokens,
        "tokens_per_question": round(total_tokens / n, 1),
        "gate_80pct_accuracy": acc >= 80,
    }
    print(variant, json.dumps(summary[variant]))
best = max(summary, key=lambda v: summary[v]["overall"])
print("WINNER (overall):", best, summary[best]["overall"])
json.dump(summary, open("/kaggle/working/phase2/summary.json", "w"), indent=2)
print("wrote /kaggle/working/phase2/summary.json")


In [ ]:
# Wrap-up: session wall time ~= GPU-hours on T4 x2.
import time
t0 = float(open("/kaggle/working/phase2/t0.txt").read())
elapsed_h = (time.time() - t0) / 3600
print(f"session wall time: {elapsed_h:.2f} h (~= GPU-hours on T4 x2, budget was 6 h)")
!df -h /kaggle/working | tail -1
!find /kaggle/working/phase2 -type f | sort
print("DONE — outputs in /kaggle/working/phase2/")
